# Assignment 2B — RAG Pipeline
*Chunking, dense / keyword / hybrid search, reranking, grounded generation and an LLM judge*

**Domain:** Medical and clinical literature (Type 2 Diabetes management)

**Generator:** Assignment 1 model — `mistralai/Mistral-7B-v0.1` + CPT checkpoint + QLoRA adapter (Assignment 1 model)

| Part | What we do | Marks |
| --- | --- | --- |
| A | Dense search and chunk size | 5 |
| B | Dense vs BM25 vs hybrid (RRF) | 4 |
| C | Cross-encoder reranking | 4 |
| D | Grounded answers and an LLM judge | 7 |
| **Total** | | **20** |

**Our choice:** the final run is on the lab server, where the Assignment 1 model files are. The brief asks for one notebook for the whole assignment, with all outputs visible.

**Models used, and where each choice comes from**

| Role | Model | Source |
| --- | --- | --- |
| Generator | `mistralai/Mistral-7B-v0.1` + CPT + QLoRA adapter (Assignment 1 model) | PDF Overview (p1): "your fine-tuned model as the answer generator" |
| Embedding | `BAAI/bge-base-en-v1.5` | PDF A0 (p1): "for example 256 tokens for all-MiniLM-L6-v2". PDF A2 (p1): "either switch to a model with a longer limit or replace that size" (we chose a model with a longer limit, as the rule allows) |
| Vector index | FAISS `IndexFlatIP` | PDF A1.1 (p1): "builds a FAISS IndexFlatIP index (or another vector index)" |
| Keyword search | BM25 with `rank-bm25` | PDF B2 (p2): "build a BM25 index". The PDF names no library; `rank-bm25` is our choice |
| Reranker | `cross-encoder/ms-marco-MiniLM-L-6-v2` | PDF C1 (p3): "for example cross-encoder/ms-marco-MiniLM-L-6-v2" |
| Judge | `Qwen/Qwen2.5-7B-Instruct` | PDF D2 (p4): "Use a judge model different from your generator, for example Llama 3.1 8B Instruct or Qwen2.5 7B Instruct, and name it" |

**Setup (Assignment 1 conventions, our choice, not in the brief).** This cell follows the Assignment 1 notebook: the same installs, the same device check, and the same Hugging Face cache. It is needed on the Kubeflow lab server.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: finds out where the notebook is running (Kubeflow, Colab or local),
# picks a folder to store the files, sets the Hugging Face cache folder and checks the GPU.
# Why it runs first: Hugging Face reads HF_HOME when transformers is imported, so it must be set before.
# Output: environment, device, dtype, GPU name and the folders used.
# ---------------------------------------------------------------------------
import importlib.util
import os
from types import SimpleNamespace


def detect_env():
    """Return "kubeflow", "colab" or "local". Kubeflow is checked first."""
    # Kubeflow sets NB_PREFIX or KUBERNETES_SERVICE_HOST, so either one means we are on the lab server.
    if ("NB_PREFIX" in os.environ or "KUBERNETES_SERVICE_HOST" in os.environ
            or os.path.isdir("/home/jovyan/data")):
        return "kubeflow"
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "colab"
    except ModuleNotFoundError:      # no "google" package at all, so this is not Colab
        pass
    return "local"


def setup():
    """Find the storage root, point the Hugging Face cache at it, and detect the GPU and dtype."""
    env = detect_env()
    # Storage root per platform (Colab's disk is wiped when the session ends).
    candidates = {"colab": ["/content"], "kubeflow": ["/home/jovyan/assessment", "/home/jovyan/data"],
                  "local": [os.path.expanduser("~/lab-data"), os.getcwd()]}[env]
    root = os.getcwd()
    # Try each folder in turn. The first one we can write to becomes the storage root.
    for path in candidates:          # first folder we can write to wins
        try:
            os.makedirs(path, exist_ok=True)
            probe = os.path.join(path, ".write_test")
            open(probe, "w").close()
            os.remove(probe)
            root = path
            break
        except Exception:
            continue

    # Model cache inside the root. Set before transformers is imported.
    cache_root = os.path.join(root, "hf-cache")
    os.environ["HF_HOME"] = cache_root
    os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # no download bars (they show up as empty lines in the notebook)

    # Device and dtype: GPU if there is one, bf16 when the GPU supports it, else fp16, else fp32 on CPU.
    # Defaults for a CPU-only run. The GPU values below replace them if a GPU is found.
    device, dtype, gpu_name, gpu_mem = "cpu", "float32", None, 0.0
    try:
        import torch
        if torch.cuda.is_available():
            device = "cuda"
            gpu_name = torch.cuda.get_device_name(0)
            gpu_mem = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
            # bf16 is used when the GPU supports it: same range as fp32, half the memory.
            dtype = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
    except Exception:
        pass

    state = SimpleNamespace(ENV=env, ROOT=root, CACHE_ROOT=cache_root, DEVICE=device,
                            DTYPE=dtype, IS_GPU=(device == "cuda"), GPU_NAME=gpu_name, GPU_MEM_GB=gpu_mem)
    print(f"env={env}  device={device}  dtype={dtype}  gpu={gpu_name or 'none'} ({gpu_mem} GB)")
    print(f"storage root : {root}")
    print(f"HF cache     : {cache_root}")
    return state


L = setup()

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: sets the input and output paths, checks that every input exists and fixes the seed.
# Why: every later step reads these paths, and a missing file should fail here, not halfway through.
# Output: OK or MISSING for each input.
# ---------------------------------------------------------------------------

import json
import random
import re
from pathlib import Path

import numpy as np

# Fixed seeds, so any random step gives the same result on every run.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Kubeflow lab layout: this notebook is /home/jovyan/assessment/Ass-2/Assignment2B.ipynb, so ../Ass-1 is
# /home/jovyan/assessment/Ass-1 (shown as /assessment/Ass-1 in the Jupyter file browser). The setup cell puts the
# Hugging Face cache in the same storage root, at /home/jovyan/assessment/hf-cache.
ASS1 = Path("../Ass-1").resolve()
CORPUS_DIR = ASS1 / "domain_corpus"                      # 14 cleaned .txt files
DATASET_PATH = ASS1 / "instruction_dataset.jsonl"        # 400 rows; the LAST 80 are the eval set
# Do NOT use ASS1 / "instruction_eval.jsonl": it holds 93 pairs, none of them in instruction_dataset.jsonl.
# The Assignment 1 B1 output says the eval split has 80 pairs (the last 80 rows of the dataset).

# Our choice, from Assignment 1 conventions (cells 29 and 49): model files are under ASS1 / "persistent".
# The prints below show where they resolve.
CPT_DIR = ASS1 / "persistent" / "cpt_ckpt"
ADAPTER_DIR = ASS1 / "persistent" / "adapter"
print(f"Assignment 1 folder : {ASS1}")
print(f"CPT checkpoint      : {CPT_DIR}")
print(f"QLoRA adapter       : {ADAPTER_DIR}")
print(f"HF cache            : {L.CACHE_ROOT}")

DATA_DIR, OUT_DIR = Path("data"), Path("outputs")
# Create the data and outputs folders if missing. exist_ok means nothing fails if they already exist.
for folder in (DATA_DIR, OUT_DIR):
    folder.mkdir(exist_ok=True)


def count_rows(path):
    """Number of non-empty lines in a JSONL file."""
    with open(path, encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())


# Each check is a label and a True/False result. Printing them shows which inputs are missing.
checks = [
    ("corpus: 14 .txt files", CORPUS_DIR.is_dir() and len(list(CORPUS_DIR.glob("*.txt"))) == 14),
    ("instruction_dataset.jsonl: 400 rows", DATASET_PATH.exists() and count_rows(DATASET_PATH) == 400),
    ("CPT checkpoint folder (lab server only)", CPT_DIR.is_dir()),
    ("QLoRA adapter folder (lab server only)", ADAPTER_DIR.is_dir()),
]
for label, ok in checks:
    print(f"{'OK' if ok else 'MISSING':8} {label}")


## Prerequisites — Environment Setup

We install the packages that are missing and print their versions, so anyone can see which versions gave these results.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: installs the packages that are missing.
# Why: the notebook should run on a fresh lab server or Colab runtime.
# Output: which packages were already there and which were installed.
# ---------------------------------------------------------------------------

import importlib.util
import subprocess
import sys

# pip name -> import name
PACKAGES = {
    "sentence-transformers": "sentence_transformers",  # embeddings (A, B) and CrossEncoder (C)
    "faiss-cpu": "faiss",                              # IndexFlatIP vector index (A, B)
    "rank-bm25": "rank_bm25",                          # BM25Okapi keyword index (B)
    "transformers": "transformers",                    # tokenizers, generator and judge (A, D)
    "peft": "peft",                                    # QLoRA adapter (D)
    "accelerate": "accelerate",                        # device_map loading (D)
    "pandas": "pandas",                                # tables
}

# Keep only the packages whose import name cannot be found in this Python.
missing = [pip for pip, module in PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    print("installing:", ", ".join(missing))
    # Use this Python's own pip, so the packages go into the environment the notebook runs in.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("all packages were already installed")
print("already there:", ", ".join(p for p in PACKAGES if p not in missing))


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines print_table(), a small helper that prints aligned tables.
# All reports in this notebook use it. Column widths are calculated from the data,
# so the columns stay aligned even when a value is long.
# ---------------------------------------------------------------------------
def print_table(headers, rows, title=None, align=None, save_as=None):
    """Print an aligned text table.

    headers   : column titles
    rows      : list of tuples, cells already formatted as strings so the caller
                controls thousands separators and dashes for values that don't apply
    title     : optional heading printed above a rule
    align     : per-column "l"/"r"; defaults to label-left, numbers-right
    save_as   : optional file name; the table is also saved as outputs/<save_as>.csv
    """
    # Default alignment: first column left, the rest right.
    align = align or ["l"] + ["r"] * (len(headers) - 1)
    # Keep the rows as a list, so they can be measured here and printed again below.
    body = list(rows)
    # Width of each column = longest cell in that column, header included.
    # Column width = the longest cell in that column, header included.
    widths = [max(len(headers[c]), max(len(r[c]) for r in body)) for c in range(len(headers))]
    gap = "   "  # space between columns
    # Full table width, used for the ruled lines.
    # Total width of the table, used for the ruled lines.
    width = sum(widths) + len(gap) * (len(widths) - 1)

    # Format one row: pad each cell to its column width.
    def line(cells):
        """Format one table row: pad each cell to its column width."""
        return gap.join(c.ljust(w) if a == "l" else c.rjust(w)
                        for c, w, a in zip(cells, widths, align))

    # Print the title, the header and the rows.
    if title:
        print("=" * width)
        print(title)
    print("=" * width)
    print(line(headers))
    print("-" * width)
    for r in rows:
        print(line(r))
    print("=" * width)

    if save_as:
        import csv
        with open(OUT_DIR / f"{save_as}.csv", "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(headers)
            writer.writerows(rows)


def show_md_table(headers, rows, title=None, save_as=None):
    """Show a table as Markdown, so long text wraps in the notebook.

    Used for tables with long text (queries, spans, judge reasons). save_as also writes outputs/<save_as>.csv.
    """
    from IPython.display import Markdown, display

    def cell(text):
        """Keep one table cell on one line, and stop a "|" from starting a new column."""
        return " ".join(str(text).split()).replace("|", "/")

    md = f"**{title}**\n\n" if title else ""
    md += "| " + " | ".join(headers) + " |\n| " + " | ".join("---" for _ in headers) + " |\n"
    md += "\n".join("| " + " | ".join(cell(c) for c in r) + " |" for r in rows)
    display(Markdown(md))
    if save_as:
        import csv
        with open(OUT_DIR / f"{save_as}.csv", "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(headers)
            writer.writerows(rows)


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines working(), a context manager that shows a status line during slow steps.
# Why: loading a 14 GB model takes minutes, and the status line shows the notebook is still running.
# Output: nothing (definition only). Each slow step prints its time when it ends.
# ---------------------------------------------------------------------------
import threading
import time
from pathlib import Path


def _folder_gb(folder):
    """Size of everything inside a folder in GB (0 if it does not exist yet)."""
    try:
        # Add up the size of each real file. Symlinks are skipped so the same file is not counted twice.
        return sum(f.stat().st_size for f in Path(folder).rglob("*") if f.is_file() and not f.is_symlink()) / 1024**3
    except OSError:
        return 0.0


class working:
    """Context manager: shows 'label ... 2m10s' (and the folder size, if given) while the code inside runs."""

    def __init__(self, label, folder=None, expected_gb=None):
        """Remember the message and, optionally, the folder to measure and its expected final size."""
        self.label, self.folder, self.expected_gb = label, folder, expected_gb

    def _line(self):
        """Build the status text for the current moment."""
        # Whole seconds since the step started, shown as minutes and seconds.
        secs = int(time.time() - self.t0)
        text = f"{self.label} ... {secs // 60}m{secs % 60:02d}s"
        if self.folder is not None:
            gb = _folder_gb(self.folder)
            text += f"  ({gb:.1f} of ~{self.expected_gb:.1f} GB)" if self.expected_gb else f"  ({gb:.1f} GB so far)"
        return text

    def _run(self):
        """Background loop: rewrite the status line every 5 seconds until the step ends."""
        while not self._stop.wait(5):
            print("\r" + self._line() + "   ", end="", flush=True)

    def __enter__(self):
        """Start the status line."""
        self.t0, self._stop = time.time(), threading.Event()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, *exc):
        """Stop the status line and print the final time."""
        self._stop.set()
        self._thread.join()
        # Change ' ...' to ' ... done in', so the last line of the status reads as the time the step took.
        print("\r" + self._line().replace(" ...", " ... done in", 1) + "   ")
        return False


## Before Part A — 10 queries and their answer spans

**Goal:** fix the 10 queries, their expected answers and their answer spans. Parts A, B and C score every search with these spans.

**What we do (our choice, not in the brief):**
1. Take the Assignment 1 eval set: the last 80 rows of `instruction_dataset.jsonl`. The dataset has no passage field, so each row is matched to the corpus.
2. For each row, find the corpus sentence (8 to 40 words) with the most word overlap with the expected answer. That sentence is the span, and its file is the passage.
3. Keep spans that occur exactly once in the corpus. Keep the best row per file and take 10 rows.
4. The expected answer is the row's `response`. The judge in D2 uses it.
5. Human check: read each span against its query, and replace any span that does not answer it.

A retrieved chunk is a **hit** if it contains the answer span.

*Our choice: the course may not accept this matching method, so confirm it before the final run.*


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: loads the last 80 eval rows, matches each answer to a corpus sentence and picks 10 queries, one per file.
# Why: the brief needs one answer span per query. The match is our choice (see the note above).
# Output: the 10 queries with their passage file and span.
# ---------------------------------------------------------------------------

eval_rows = [json.loads(line) for line in open(DATASET_PATH, encoding="utf-8")][-80:]
assert len(eval_rows) == 80

STOP = set("the a an of and or to in on for with by is are was were be as at from that this it its which".split())

# Lowercase words of letters and digits, minus the stop words, as a set.
def tokens(text):
    return set(re.findall(r"[a-z0-9]+", text.lower())) - STOP

# Candidate spans: corpus sentences of 8 to 40 words, with their file.
SENTENCES = []
for f in sorted(CORPUS_DIR.glob("*.txt")):
    text = " ".join(f.read_text(encoding="utf-8").split())
    for sent in re.split(r"(?<=[.!?])\s+", text):
        if 8 <= len(sent.split()) <= 40:
            SENTENCES.append((f.name, sent))

def best_span(response):
    """Score = overlap / sqrt(sentence length). Ties go to the shorter sentence."""
    want = tokens(response)
    scored = []
    for source, sent in SENTENCES:
        have = tokens(sent)
        if have:
            overlap = len(want & have)
            # max() compares the score first, then the overlap, then -length, so a tie goes to the shorter sentence.
            # Score = shared words, scaled down by the square root of the sentence length, so long sentences do not win by size.
            scored.append((overlap / len(have) ** 0.5, overlap, -len(sent), source, sent))
    return max(scored)

# For every eval row, keep its best corpus sentence (the span candidate) and its file.
matches = []
for row in eval_rows:
    score, overlap, _, source, sent = best_span(row["response"])
    matches.append({"overlap": overlap, "source": source, "span": sent,
                    "query": row["instruction"], "expected_answer": row["response"]})

# Keep only spans that occur exactly once in the whole corpus, then the best match per file.
def norm(t): return " ".join(t.lower().split())
CORPUS_NORM = norm(" ".join(f.read_text(encoding="utf-8") for f in CORPUS_DIR.glob("*.txt")))
# A span that appears more than once in the corpus is ambiguous, so only spans found once are kept.
unique = [m for m in sorted(matches, key=lambda m: -m["overlap"]) if CORPUS_NORM.count(norm(m["span"])) == 1]
best_per_file = {}
for m in unique:
    # One query per file: the first match for each file is the one with the highest overlap.
    best_per_file.setdefault(m["source"], m)
QUERIES = []
# Take the 10 best matches, number them Q1 to Q10 and store the query, answer and span.
for qid, m in enumerate(sorted(best_per_file.values(), key=lambda m: -m["overlap"])[:10], start=1):
    QUERIES.append({"qid": f"Q{qid}", "source_file": m["source"], "query": m["query"],
                    "expected_answer": m["expected_answer"], "span": m["span"]})

assert len(QUERIES) == 10 and len({q["source_file"] for q in QUERIES}) == 10
print(f"eval rows: {len(eval_rows)}, candidate sentences: {len(SENTENCES):,}, queries: {len(QUERIES)}")


**Human check (our choice, not in the brief).** Word overlap can pick a sentence that shares words without answering the question. Each span below was read against its query. Q9 and Q10 matched the wrong file: their answers are lines of the "Lifestyle Goals in Diabetes" list in the ICMR guideline, so those lines are their spans. Read the 10 spans again before the final run, and change `HUMAN_CHECK` if you disagree.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: applies the human check: "ok" keeps the span, a sentence replaces it.
# Why: word overlap can pick a sentence that shares words without answering the question.
# Output: one line naming the replaced spans. Cell 13 shows the 10 queries.
# ---------------------------------------------------------------------------

HUMAN_CHECK = {
    "Q1": "ok", "Q2": "ok", "Q3": "ok", "Q4": "ok", "Q5": "ok", "Q6": "ok", "Q7": "ok", "Q8": "ok",
    "Q9": "Advise other behavioural changes for: smoking, other tobacco products and alcohol",
    "Q10": "To individualise the diet according to complications and co-morbidities",
}

# Replace each span marked other than 'ok'. The new text must be in the corpus exactly once.
for qid, value in HUMAN_CHECK.items():
    if value != "ok":
        q = next(x for x in QUERIES if x["qid"] == qid)
        assert CORPUS_NORM.count(norm(value)) == 1, f"{qid}: the replacement must occur exactly once in the corpus"
        q["span"] = value
        # Point the query at the file that holds the replacement span.
        q["source_file"] = next(f.name for f in CORPUS_DIR.glob("*.txt") if norm(value) in norm(f.read_text(encoding="utf-8")))

assert set(HUMAN_CHECK) == {q["qid"] for q in QUERIES}, "every query needs a human check"
replaced = [qid for qid, value in HUMAN_CHECK.items() if value != "ok"]
print(f"human check: spans replaced for {', '.join(replaced) or 'none'}; the other spans are kept")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: saves the 10 queries and shows them.
# Why: data/answer_spans.json is what the hit rates in Parts A, B and C use.
# Output: data/answer_spans.json and a table of the 10 queries.
# ---------------------------------------------------------------------------

# A chunk is a hit if the span text appears in it, ignoring case and extra spaces (norm).
def is_hit(chunk_text, span):
    """True if the chunk contains the answer span."""
    return norm(span) in norm(chunk_text)

# Save the queries, so Parts A to D all score against the same spans.
(DATA_DIR / "answer_spans.json").write_text(json.dumps(QUERIES, indent=1, ensure_ascii=False), encoding="utf-8")
show_md_table(["qid", "source file", "query", "expected answer", "answer span"],
              [(q["qid"], q["source_file"], q["query"], q["expected_answer"], q["span"]) for q in QUERIES],
              title="The 10 queries, their expected answers and answer spans", save_as="query_table")

## Part A — Dense search and chunk size (5 marks)

**Goal:** build dense search with FAISS and choose the chunk size for Parts B to D.

**Embedding model.** `all-MiniLM-L6-v2` cuts input at 256 tokens, so 400-token chunks would be truncated. PDF A2 allows this: "either switch to a model with a longer limit or replace that size". We chose a model with a longer limit, as the rule allows: `BAAI/bge-base-en-v1.5` (512-token limit), so both sizes fit. Tokens are counted with this model's tokenizer.

**What we do:**
1. A1.1 — one reusable dense search function (FAISS `IndexFlatIP`).
2. A2.1 — chunk the corpus at 100 and 400 tokens, run the 10 queries, fill the table.
3. A2.2 — choose the chunk size.
4. A1.2 — report the setup for the final chunk size.

*Note on the brief:* the second row of the A2 table says 500, but the A2 text and the submission checklist say 400, so we run 100 and 400.


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: loads the embedding model and prints its limit and vector size.
# Why: the brief asks us to note the model's maximum input length and count tokens with its tokenizer.
# Output: model name, max_seq_length, vector size.
# ---------------------------------------------------------------------------

from sentence_transformers import SentenceTransformer
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()    # hides harmless warnings, such as the long-file note when A2 tokenizes whole files
hf_logging.disable_progress_bar()   # no "Loading checkpoint shards" bars in the saved notebook

EMBED_MODEL_ID = "BAAI/bge-base-en-v1.5"   # Our choice. PDF A0 gives all-MiniLM-L6-v2 (256 tokens) only as an example.
# PDF A2 (p1): "either switch to a model with a longer limit or replace that size".
# We chose a model with a longer limit, as the rule allows: bge (512 tokens).
EMBED_BATCH = 64

with working("Loading the embedding model"):
    embed_model = SentenceTransformer(EMBED_MODEL_ID, device=L.DEVICE)
embed_tok = embed_model.tokenizer          # counts the chunk tokens for A2
EMBED_DIM = embed_model.get_sentence_embedding_dimension()
print(f"model              : {EMBED_MODEL_ID}")
print(f"max_seq_length     : {embed_model.max_seq_length} tokens")
print(f"vector size        : {EMBED_DIM}")


### A1.1 Dense search function (1 mark)

One function embeds the chunks, builds a FAISS `IndexFlatIP` index and returns the top k chunks for each query. Embeddings are L2-normalised, so the inner product equals cosine similarity. The index is cached per chunk set, so Parts B and C reuse it.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines dense_search(), the one reusable dense search function.
# Why: Parts A, B and C all call it, so every method is searched the same way.
# Output: nothing (definition only). The index build time is printed in A1.2.
# ---------------------------------------------------------------------------

import time

import faiss

_INDEX_CACHE = {}   # (id(chunks), len(chunks)) -> (chunks, FAISS index, build seconds)


def dense_search(chunks, queries, k):
    """Embed the chunks once (cached), build a FAISS IndexFlatIP, and return the top k for each query.

    Returns one list per query. Each item is a dict with rank (from 1), chunk_id, score and text.
    """
    # The 100- and 400-token chunk lists are different objects, so each one gets its own index.
    key = (id(chunks), len(chunks))
    if key not in _INDEX_CACHE:
        t0 = time.perf_counter()
        # Embed every chunk. normalize_embeddings makes each vector length 1, so inner product = cosine.
        vectors = embed_model.encode([c["text"] for c in chunks], batch_size=EMBED_BATCH,
                                     normalize_embeddings=True, convert_to_numpy=True).astype("float32")
        # Flat index: compares the query with every chunk, so the search is exact (no approximation).
        index = faiss.IndexFlatIP(vectors.shape[1])
        index.add(vectors)
        # Keep the index, so the same chunk set is not embedded again in each part.
        _INDEX_CACHE[key] = (chunks, index, time.perf_counter() - t0)
    _, index, _ = _INDEX_CACHE[key]

    # Queries get the same normalisation, so their inner product is also cosine similarity.
    query_vectors = embed_model.encode(queries, batch_size=EMBED_BATCH,
                                       normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    # FAISS returns the score and the chunk position of the top k for each query vector.
    scores, ids = index.search(query_vectors, k)
    results = []
    for qi in range(len(queries)):
        # Ranks start at 1. FAISS marks empty slots with -1 (fewer than k chunks), so those are skipped.
        results.append([{"rank": r + 1, "chunk_id": chunks[j]["chunk_id"], "score": float(s), "text": chunks[j]["text"]}
                        for r, (s, j) in enumerate(zip(scores[qi], ids[qi])) if j >= 0])
    return results


def index_build_seconds(chunks):
    """Seconds spent encoding the chunks and calling index.add for this chunk set."""
    if (id(chunks), len(chunks)) not in _INDEX_CACHE:
        dense_search(chunks, [QUERIES[0]["query"]], 1)
    return _INDEX_CACHE[(id(chunks), len(chunks))][2]
print("dense_search() defined: embeds the chunks, builds a FAISS IndexFlatIP index, returns the top k.")


### A2.1 Chunk size test (2 marks)

Fixed-size chunks with no overlap, per document, counted with the embedding tokenizer. The text of each chunk is sliced from the original document with the tokenizer's offsets, so the generator sees the original wording.

| Column | How |
| --- | --- |
| Total chunks | number of chunks over the 14 documents |
| Hit rate@3 (%) | share of the 10 queries where one of the top 3 chunks contains the span |
| Tokens sent per query (top 3) | combined `n_tokens` of the top 3, averaged over the 10 queries |

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines chunk_corpus(), fixed-size token chunking with no overlap.
# Why: the brief asks for 100- and 400-token chunks counted with the embedding model's tokenizer.
# Output: nothing (definition only).
# ---------------------------------------------------------------------------

def chunk_corpus(size):
    """Split each .txt file into chunks of `size` tokens with no overlap.

    Tokens are counted with the embedding tokenizer. Each chunk's text is cut from the original
    file with the token offsets, so the wording is not lowercased.
    """
    chunks = []
    for f in sorted(CORPUS_DIR.glob("*.txt")):
        text = f.read_text(encoding="utf-8")
        # offset_mapping gives the (start, end) character position of every token in the file.
        offsets = embed_tok(text, add_special_tokens=False, return_offsets_mapping=True)["offset_mapping"]
        # Step through the tokens in blocks of `size`. The last block can be shorter.
        for start in range(0, len(offsets), size):
            part = offsets[start:start + size]
            # Cut from the first token's start to the last token's end, so the chunk keeps the original text.
            chunks.append({"chunk_id": f"{f.stem}#{start // size}", "doc": f.name,
                           "text": text[part[0][0]:part[-1][1]], "n_tokens": len(part)})
    return chunks
print("chunk_corpus() defined: fixed-size token chunks, no overlap.")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: runs dense search on the 10 queries for each chunk size and fills the A2 table.
# Why: this is the evidence for choosing the chunk size.
# Output: the A2 table (chunks, hit rate@3, tokens sent per query).
# ---------------------------------------------------------------------------

A2_SIZES = [100, 400]       # the sizes in the A2 text and the submission checklist
A2_HIT = {}                  # chunk size -> hit rate@3 (%)
A2_TOKENS = {}               # chunk size -> tokens sent per query (top 3)
chunk_sets = {}              # chunk size -> list of chunks
a2_rows = []
for size in A2_SIZES:
    chunks_for_size = chunk_corpus(size)
    assert max(c["n_tokens"] for c in chunks_for_size) <= size
    chunk_sets[size] = chunks_for_size
    by_id = {c["chunk_id"]: c for c in chunks_for_size}
    # Top 3 chunks for each of the 10 queries at this chunk size.
    tops = dense_search(chunks_for_size, [q["query"] for q in QUERIES], 3)
    # A query is a hit if any of its top 3 chunks contains the answer span.
    hits = sum(any(is_hit(r["text"], q["span"]) for r in top) for q, top in zip(QUERIES, tops))
    # Tokens sent: the 3 chunk sizes added up per query, then averaged over the 10 queries.
    tokens = sum(sum(by_id[r["chunk_id"]]["n_tokens"] for r in top) for top in tops) / len(QUERIES)
    # Hit rate@3 as a percentage of the 10 queries.
    A2_HIT[size] = 100 * hits / len(QUERIES)
    A2_TOKENS[size] = tokens
    a2_rows.append((str(size), f"{len(chunks_for_size):,}", f"{A2_HIT[size]:.0f}", f"{tokens:.0f}"))
print_table(["Chunk size (tokens)", "Total chunks", "Hit rate@3 (%)", "Tokens sent per query (top 3)"],
            a2_rows, title="A2 chunk size test (top 3, 10 queries)", save_as="a2_chunk_size")



### A2.2 Chosen chunk size (1 mark)

*Inference cell below: two or three sentences built from the A2 numbers. Weigh hit rate, tokens per query, and how much context the Assignment 1 model can handle: its adapter was trained with `SFT_MAX_LEN = 512` (Assignment 1, cell 49).*

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: sets the chosen chunk size and measures the Part D prompt length with the Mistral tokenizer.
# Why: the generator counts tokens differently from the embedding model.
# Output: the chosen size and the prompt length in Mistral tokens.
# ---------------------------------------------------------------------------

from transformers import AutoTokenizer

# Our choice: the size with the highest hit rate@3. A tie goes to the smaller size, which sends fewer tokens.
# min over (-hit rate, size): the lowest key wins, so the highest hit rate wins and a tie goes to the smaller size.
CHUNK_SIZE = min(A2_SIZES, key=lambda s: (-A2_HIT[s], s))
# The chunk list for the chosen size is reused in Parts B, C and D.
chunks = chunk_sets[CHUNK_SIZE]
print(f"chosen chunk size: {CHUNK_SIZE} tokens ({len(chunks):,} chunks)")

# Our check: count one Part D prompt (3 chunks and a question) with the Mistral tokenizer from the adapter folder.
# The layout follows GEN_PROMPT in D1.1, so the count is close to the real prompt.
mistral_tok = AutoTokenizer.from_pretrained(ADAPTER_DIR)
top3 = dense_search(chunks, [QUERIES[0]["query"]], 3)[0]
# Build a prompt-shaped text: the 3 top chunks, numbered like the generator sees them, then the question.
prompt_text = "\n".join(f"[{r['rank']}] {r['text']}" for r in top3) + f"\n\nQuestion: {QUERIES[0]['query']}"
# Count tokens with the generator's own tokenizer, since the 512 limit is for the generator.
PROMPT_TOKENS = len(mistral_tok(prompt_text).input_ids)
print(f"Mistral tokens for one prompt (3 chunks and a question): {PROMPT_TOKENS}")
print("adapter training length: 512 tokens (SFT_MAX_LEN, Assignment 1)")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: names the chunk size used in Parts B to D, and why, from the A2 numbers.
# Why: PDF A2.2 asks us to name the chunk size and why, and to consider hit rate, tokens per query and the model's context.
# PDF step: A2.2 (p2). Output: one paragraph.
# ---------------------------------------------------------------------------

from IPython.display import Markdown, display

fits = "within" if PROMPT_TOKENS <= 512 else "above"
display(Markdown(
    f"**A2.2 chosen size: {CHUNK_SIZE} tokens, used in Parts B, C and D.** Hit rate@3 was {A2_HIT[100]:.0f}% at "
    f"100 tokens and {A2_HIT[400]:.0f}% at 400, and the top 3 sent {A2_TOKENS[100]:.0f} and {A2_TOKENS[400]:.0f} "
    f"tokens per query. One Part D prompt at {CHUNK_SIZE} tokens is {PROMPT_TOKENS} Mistral tokens, which is {fits} "
    f"the 512 tokens the Assignment 1 adapter was trained on."))


### A1.2 Dense search setup (1 mark)

Reported for the final chunk set chosen in A2.2.

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: prints the dense search setup for the final chunk set.
# Why: the brief asks for these numbers for the final chunk set.
# Output: a table with the model, vector size, similarity, batch size, chunk count and build time.
# ---------------------------------------------------------------------------

print_table(["Setting", "Value"], [
    ("Embedding model", EMBED_MODEL_ID),
    ("Vector size", str(EMBED_DIM)),
    ("Similarity measure", "inner product on L2-normalised vectors (cosine)"),
    ("Batch size", str(EMBED_BATCH)),
    ("Number of chunks", f"{len(chunks):,}"),
    # Build time is the value stored when the index was built in dense_search.
    ("Index build time (s)", f"{index_build_seconds(chunks):.2f}"),
], title=f"A1.2 dense search setup, chunk size {CHUNK_SIZE} tokens", save_as="a1_2_setup")


## Part B — Keyword and hybrid search (4 marks)

**Goal:** compare dense search, BM25 and a hybrid of the two on the same 10 queries, top 5 each.

**What we do:**
1. B1 — dense search from Part A.
2. B2 — BM25 index, top 5.
3. B3 — hybrid with Reciprocal Rank Fusion over the top 20 of each method:
   `RRF(d) = 1 / (60 + rank in dense) + 1 / (60 + rank in BM25)`. Ranks start at 1; a chunk missing from a list gets nothing from it.
4. B4 — which method works best, with one example query.

| Metric | How |
| --- | --- |
| Query latency (ms) | time to return the top 5 for one query, averaged over the 10 queries, after one warm-up run |
| Hit rate@1 (%) | share of queries whose first result contains the span |
| Hit rate@3 (%) | share of queries with the span in the top 3 |

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines the shared evaluate() helper used for every method in Parts B and C.
# Why: one helper means the rows of the table are measured the same way.
# Output: nothing (definition only).
# ---------------------------------------------------------------------------

def evaluate(search_fn, k=5):
    """Run search_fn(query, k) on the 10 queries after one warm-up.

    Returns avg latency (ms), hit rate@1 (%), hit rate@3 (%) and the ranked lists per query.
    """
    # One warm-up call first, so model loading and caches do not inflate the first timed query.
    search_fn(QUERIES[0]["query"], k)   # warm-up, not timed
    latencies, ranked = [], {}
    for q in QUERIES:
        # Time only the search call for this query.
        t0 = time.perf_counter()
        ranked[q["qid"]] = search_fn(q["query"], k)
        latencies.append((time.perf_counter() - t0) * 1000)

    # Hit rate@n: a query counts once if any of its top n results contains the answer span.
    def hit_at(n):
        """Share of queries whose top n contain the answer span, in %."""
        found = sum(any(is_hit(r["text"], q["span"]) for r in ranked[q["qid"]][:n]) for q in QUERIES)
        return 100 * found / len(QUERIES)

    # Average time per query, plus the two hit rates. The ranked lists are kept for the rank table.
    return {"latency_ms": sum(latencies) / len(latencies), "hit1": hit_at(1), "hit3": hit_at(3), "ranked": ranked}
print("evaluate() defined: average latency, hit rate@1 and hit rate@3 for one search method.")


### B1 Dense (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: runs dense search (top 5) through evaluate().
# Why: fills the Dense row of the B table.
# Output: latency, hit rate@1, hit rate@3 for Dense.
# ---------------------------------------------------------------------------

b_results = {}   # method name -> evaluate() result
# Wrap dense search so it has the same (query, k) signature as the BM25 and hybrid functions.
dense_fn = lambda q, k: dense_search(chunks, [q], k)[0]
b_results["Dense"] = evaluate(dense_fn)
print(f"Dense: hit rate@1 {b_results['Dense']['hit1']:.0f}%, hit rate@3 {b_results['Dense']['hit3']:.0f}%")


### B2 BM25 (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: builds a BM25 index over the chunks and runs it (top 5).
# Why: keyword search catches exact terms such as drug names and HbA1c.
# Output: latency, hit rate@1, hit rate@3 for BM25.
# ---------------------------------------------------------------------------

# Our choice, not in the PDF: the rank-bm25 library. The PDF asks for a BM25 index and names no library.
from rank_bm25 import BM25Okapi


def bm25_tokens(text):
    """Lowercase words and numbers. Keeps terms like hba1c and sglt2 whole."""
    # Keep letters and digits together, so codes such as hba1c and sglt2 stay one term.
    return re.findall(r"[a-z0-9]+", text.lower())


# Build the BM25 index over the same chunk list, one tokenised list per chunk.
bm25 = BM25Okapi([bm25_tokens(c["text"]) for c in chunks])


def bm25_fn(query, k):
    """Top k chunks for one query, in the same format as dense_search."""
    # One BM25 score per chunk for this query.
    scores = bm25.get_scores(bm25_tokens(query))
    # Highest BM25 scores first.
    top = np.argsort(-scores)[:k]
    return [{"rank": r + 1, "chunk_id": chunks[j]["chunk_id"], "score": float(scores[j]), "text": chunks[j]["text"]}
            for r, j in enumerate(top)]


b_results["BM25"] = evaluate(bm25_fn)
print(f"BM25: hit rate@1 {b_results['BM25']['hit1']:.0f}%, hit rate@3 {b_results['BM25']['hit3']:.0f}%")


### B3 Hybrid with RRF (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines rrf_fuse() and runs the hybrid search (top 5).
# Why: RRF combines the two ranked lists without needing their scores to be on the same scale.
# Output: the Hybrid row.
# ---------------------------------------------------------------------------


def rrf_fuse(dense_ids, bm25_ids, k=60, top=5):
    """Reciprocal rank fusion: score = sum of 1 / (k + rank) over the lists the chunk appears in."""
    score = {}
    for ids in (dense_ids, bm25_ids):
        # Ranks start at 1. A chunk that is missing from a list gets nothing from that list.
        for rank, cid in enumerate(ids, start=1):
            # Add 1 / (k + rank) for each list the chunk is in. With k = 60, a rank gap matters less at the top.
            score[cid] = score.get(cid, 0.0) + 1.0 / (k + rank)
    return sorted(score.items(), key=lambda item: -item[1])[:top]


chunk_by_id = {c["chunk_id"]: c for c in chunks}


def hybrid_fn(query, k):
    """Top 20 from dense and from BM25, fused with RRF, keep the top k."""
    # Take the top 20 from each method before fusing, so a chunk ranked well by both can rise.
    dense_top = dense_fn(query, 20)
    bm25_top = bm25_fn(query, 20)
    # Fuse the two lists of chunk ids, then keep the top k.
    fused = rrf_fuse([r["chunk_id"] for r in dense_top], [r["chunk_id"] for r in bm25_top], top=k)
    return [{"rank": r + 1, "chunk_id": cid, "score": s, "text": chunk_by_id[cid]["text"]}
            for r, (cid, s) in enumerate(fused)]


b_results["Hybrid (RRF)"] = evaluate(hybrid_fn)
print(f"Hybrid (RRF): hit rate@1 {b_results['Hybrid (RRF)']['hit1']:.0f}%, "
      f"hit rate@3 {b_results['Hybrid (RRF)']['hit3']:.0f}%")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: prints the B comparison table (the Part B table in the brief).
# Why: the brief asks for this table, filled for all three methods.
# Output: the B table and the per-query rank table.
# ---------------------------------------------------------------------------


def rank_of(method, qid, span):
    """Rank of the first top-5 chunk that contains the span, or "-" if none does."""
    # Walk the ranked list from the top, so the first hit is the best rank.
    for r in b_results[method]["ranked"][qid]:
        if is_hit(r["text"], span):
            return str(r["rank"])
    return "-"


print_table(["Method", "Average query latency (ms)", "Hit rate@1 (%)", "Hit rate@3 (%)"],
            [(m, f"{v['latency_ms']:.1f}", f"{v['hit1']:.0f}", f"{v['hit3']:.0f}") for m, v in b_results.items()],
            title=f"Part B search comparison (chunk size {CHUNK_SIZE}, top 5)", save_as="b_comparison")


### B4 Best method (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: Inference - B4: which method works best and why, with one example query.
# Why: the text is built from the B table, so it stays in line with the numbers.
# Output: one paragraph.
# ---------------------------------------------------------------------------

from IPython.display import Markdown, display

# Our choice: best hit rate@1, then hit rate@3, then the faster method.
# If all methods tie on hit rate@1 and @3, the fastest one is picked.
# max over (hit rate@1, hit rate@3, -latency): the best hit rate wins, and speed breaks ties.
BEST_METHOD = max(b_results, key=lambda m: (b_results[m]["hit1"], b_results[m]["hit3"], -b_results[m]["latency_ms"]))
REASON = {
    "Dense": "dense search matches meaning, so it finds answers that are worded differently from the question",
    "BM25": "BM25 matches exact words such as drug names and numbers, which dense search can blur",
    "Hybrid (RRF)": "the hybrid keeps a chunk that either list ranks well, so it recovers answers that one method misses",
}
# PDF step: B4 (p3), one query as an example.
# Our choice: the example is a query where the best method ranks the answer chunk strictly higher than
# every other method (a missing chunk counts as the worst rank), so the example supports the choice.
# If no query does this, the first query where the ranks differ is used.
def _rank_or_last(method, q):
    r = rank_of(method, q["qid"], q["span"])
    return int(r) if r.isdigit() else 99


ex = next((q for q in QUERIES
           if _rank_or_last(BEST_METHOD, q) < min(_rank_or_last(m, q) for m in b_results if m != BEST_METHOD)), None)
if ex is None:
    ex = next((q for q in QUERIES if len({rank_of(m, q["qid"], q["span"]) for m in b_results}) > 1), QUERIES[0])
# Shortcut to the chosen method's numbers, used in the text below.
best = b_results[BEST_METHOD]
others = " and ".join(f"{m} {b_results[m]['hit1']:.0f}% / {b_results[m]['hit3']:.0f}%" for m in b_results if m != BEST_METHOD)
# Methods that match the best on both hit rates, listed for the tie note.
tied = [m for m in b_results if m != BEST_METHOD and (b_results[m]["hit1"], b_results[m]["hit3"]) == (best["hit1"], best["hit3"])]
tie_note = (f" It ties with {' and '.join(tied)} on both hit rates and is picked because it is faster "
            f"({best['latency_ms']:.1f} ms per query)." if tied else "")
display(Markdown(
    f"**B4 best method: {BEST_METHOD}.** Hit rate@1 / hit rate@3: {best['hit1']:.0f}% / {best['hit3']:.0f}%, "
    f"against {others}.{tie_note} Why: {REASON[BEST_METHOD]}. Example query {ex['qid']}: "
    f"\"{ex['query']}\" The answer chunk ranks {rank_of('Dense', ex['qid'], ex['span'])} with Dense, "
    f"{rank_of('BM25', ex['qid'], ex['span'])} with BM25 and {rank_of('Hybrid (RRF)', ex['qid'], ex['span'])} "
    f"with Hybrid (RRF)."))


## Part C — Reranking (4 marks)

**Goal:** re-score the top 10 of the best Part B method with a cross-encoder and keep the top 3.

**What we do:**
1. C1 — retrieve the top 10 with the best method; re-score with `cross-encoder/ms-marco-MiniLM-L-6-v2`; keep the top 3.
2. C2 — show the top 3 before and after reranking for each query, and the rank change rate.
3. C3 — reranking time per query and hit rate@1 before and after.
4. C4 — did reranking help?

**Rank change rate** = share of the 10 queries whose first result changes after reranking.

### C1 Rerank (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: loads the cross-encoder and reranks the top 10 of the best method for each query.
# Why: a cross-encoder reads the query and the chunk together, so it can order them better.
# Output: the device used and final_chunks (top 3 per query).
# ---------------------------------------------------------------------------

import torch
from sentence_transformers import CrossEncoder

RERANK_MODEL_ID = "cross-encoder/ms-marco-MiniLM-L-6-v2"   # PDF C1: "for example cross-encoder/ms-marco-MiniLM-L-6-v2". Any cross-encoder is allowed; we chose this one.
with working("Loading the cross-encoder"):
    # The cross-encoder scores (query, chunk) pairs together, so it reads both at once.
    ce = CrossEncoder(RERANK_MODEL_ID, device=L.DEVICE)
print(f"cross-encoder: {RERANK_MODEL_ID} on {L.DEVICE}")

# Map the best method's name to its search function, so reranking uses that method.
best_fn = {"Dense": dense_fn, "BM25": bm25_fn, "Hybrid (RRF)": hybrid_fn}[BEST_METHOD]


def rerank_top3(query):
    """Top 10 with the best method, reranked. Returns (before top 3, after top 3, predict seconds)."""
    top10 = best_fn(query, 10)
    # One (query, chunk) pair per candidate, scored in a single batch.
    pairs = [(query, c["text"]) for c in top10]
    t0 = time.perf_counter()
    scores = ce.predict(pairs)
    if L.IS_GPU:
        torch.cuda.synchronize()    # GPU work is asynchronous, so wait before stopping the timer
    seconds = time.perf_counter() - t0
    # Highest cross-encoder score first; keep the top 3 of the 10.
    # Sort the 10 candidates by cross-encoder score, highest first, and keep the top 3 positions.
    order = sorted(range(len(top10)), key=lambda i: -scores[i])[:3]
    # Store the reranked chunks with their new rank and the cross-encoder score.
    after = [dict(top10[i], rank=r + 1, ce_score=float(scores[i])) for r, i in enumerate(order)]
    return top10[:3], after, seconds


ce.predict([(QUERIES[0]["query"], "warm-up")])   # warm-up, not timed
# Run the rerank for each query. before_top3 keeps the first 3 from the best method for the comparison.
final_chunks, before_top3, rerank_ms = {}, {}, []
for q in QUERIES:
    before, after, seconds = rerank_top3(q["query"])
    before_top3[q["qid"]] = before
    final_chunks[q["qid"]] = after
    rerank_ms.append(seconds * 1000)
print(f"reranked {len(QUERIES)} queries; kept the top 3 of 10 for each")


### C2 Compare rankings (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: shows the top 3 before and after reranking for each query.
# Why: the brief asks to see the change for every query.
# Output: the before and after top 3 per query, and the rank change rate.
# ---------------------------------------------------------------------------

changed = 0
for q in QUERIES:
    qid = q["qid"]
    print(f"\n{qid}  {q['query']}")
    print("  Before reranking (best method's own top 3):")
    for r in before_top3[qid]:
        # HIT marks a chunk that contains the answer span, so the change in the ranking is easy to see.
        mark = "HIT " if is_hit(r["text"], q["span"]) else "    "
        print(f"    {r['rank']}. {mark}{r['chunk_id']:<30} {' '.join(r['text'].split())[:80]}")   # one line per chunk
    print("  After reranking (cross-encoder top 3):")
    for r in final_chunks[qid]:
        mark = "HIT " if is_hit(r["text"], q["span"]) else "    "
        print(f"    {r['rank']}. {mark}{r['chunk_id']:<30} {' '.join(r['text'].split())[:80]}")
    # Rank change: the first result is a different chunk after reranking.
    # Rank change: the top result changed if its chunk id differs after reranking.
    if before_top3[qid][0]["chunk_id"] != final_chunks[qid][0]["chunk_id"]:
        changed += 1

RANK_CHANGE = 100 * changed / len(QUERIES)   # % of queries whose first result changed
print(f"\nRank change rate: {RANK_CHANGE:.0f}% of queries (first result changed)")


### C3 Measure (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: fills the reranking table.
# Why: weighs accuracy against the time reranking adds.
# Output: reranking time per query, rank change rate, hit rate@1 before and after.
# ---------------------------------------------------------------------------

hit1_before = b_results[BEST_METHOD]["hit1"]
# Hit rate@1 after reranking: share of queries whose new first chunk contains the span.
hit1_after = 100 * sum(is_hit(final_chunks[q["qid"]][0]["text"], q["span"]) for q in QUERIES) / len(QUERIES)
# Average rerank time per query, in ms.
RERANK_MS = sum(rerank_ms) / len(rerank_ms)
print_table(["Measure", "Value"], [
    ("Reranking time per query (ms)", f"{RERANK_MS:.1f}"),
    ("Rank change rate (%)", f"{RANK_CHANGE:.0f}"),
    ("Hit rate@1 before reranking (from Part B, same method)", f"{hit1_before:.0f}"),
    ("Hit rate@1 after reranking", f"{hit1_after:.0f}"),
], title="Part C reranking measures", save_as="c_measures")


### C4 Did reranking help? (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: Inference - C4: whether reranking helped, from the hit rates and the time added.
# Why: the text is built from the C3 numbers, so it stays in line with them.
# Output: one paragraph.
# ---------------------------------------------------------------------------

from IPython.display import Markdown, display

# Compare the hit rate@1 before and after to pick the wording of the verdict.
if hit1_after > hit1_before:
    verdict = "reranking helped"
elif hit1_after == hit1_before:
    verdict = "reranking made no difference"
else:
    verdict = "reranking lowered hit rate@1"
note = ("" if hit1_after > hit1_before else
        " The cross-encoder was trained on web search (MS MARCO), not clinical text, which may explain this.")
display(Markdown(
    f"**C4.** Hit rate@1 went from {hit1_before:.0f}% to {hit1_after:.0f}% for +{RERANK_MS:.0f} ms per query, "
    f"so {verdict}.{note} "
    f"Weighing accuracy against time: the hit rate@1 gain of {hit1_after - hit1_before:.0f} points costs {RERANK_MS:.0f} ms per query."))


## Part D — Generate and check answers (7 marks)

### D1 Generate grounded answers (3 marks)

**Goal:** answer the 10 queries with the Assignment 1 model, using only the 3 reranked chunks, with citations and a "Not found" reply.

**What we do:**
The brief has three D1 steps, one mark each:
1. D1.1 — the prompt (1 mark): answer only from the chunks, cite them, and say "Not found in the documents".
2. D1.2 — all 10 answers (1 mark), each from the 3 reranked chunks.
3. D1.3 — one query whose answer is not in the corpus (1 mark).

**Generator:** the Assignment 1 model only (Mistral-7B-v0.1 + CPT weights + QLoRA adapter). It answers the 10 queries and the out-of-corpus query.


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: loads the Assignment 1 generator: the CPT model plus the QLoRA adapter. The length settings are our choice, copied from Assignment 1 cell 51.
# Why: the brief asks us to reuse the Assignment 1 model as the answer generator.
# Output: the generator name and the loaded model.
# ---------------------------------------------------------------------------

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

GEN_BASE_ID = "mistralai/Mistral-7B-v0.1"   # PDF Overview: "your fine-tuned model as the answer generator" (Assignment 1 MODEL_ID, cell 18).
GENERATOR_NAME = f"{GEN_BASE_ID} + CPT + QLoRA adapter (Assignment 1)"
GEN_MAX_NEW_TOKENS = 120                    # Our choice, not in the PDF: copied from Assignment 1 SFT_MAX_NEW_TOKENS (cell 51): answers are 1-3 sentences
GEN_STOPS = ["Instruction:", "Question:"]   # Our choice, not in the PDF: the next prompt starts here, so the answer ends here

with working("Loading the generator (CPT model, then the adapter)", folder=CPT_DIR):
    # Load the CPT checkpoint on GPU 0.
    gen_base = AutoModelForCausalLM.from_pretrained(CPT_DIR, device_map={"": 0})
    # Add the QLoRA adapter on top of the CPT model. eval() turns off dropout for answering.
    gen_model = PeftModel.from_pretrained(gen_base, ADAPTER_DIR).eval()
# The tokenizer saved with the adapter, which has the chat template from Assignment 1.
gen_tok = AutoTokenizer.from_pretrained(ADAPTER_DIR)   # saved with the chat template in Assignment 1 (cells 49 and 50)
print(f"generator: {GENERATOR_NAME}")


#### D1.1 Generation prompt (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines the generation prompt and prints it.
# Why: the brief asks to show the prompt (D1.1, p3).
# Output: the prompt template.
# ---------------------------------------------------------------------------

# A plain template. generate_answer() fills in the chunks and the question.
GEN_PROMPT = """Answer the question using ONLY the numbered chunks below.
Cite the chunk number you used after each fact, for example [2]. Use only the numbers 1, 2 and 3.
Stop after your answer. Do not write a new question or a new instruction.
If the chunks do not contain the answer, reply exactly: Not found in the documents

Chunks:
[1] {chunk_1}
[2] {chunk_2}
[3] {chunk_3}

Question: {question}"""

print(GEN_PROMPT)


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines generate_answer(), clean_answer() and the citation and Not found checks.
# Why: the same code answers the 10 queries and the out-of-corpus query.
# Output: nothing (definitions only).
# ---------------------------------------------------------------------------

NOT_FOUND = "Not found in the documents"
CITE = re.compile(r"\[(\d+)\]")


def clean_answer(raw):
    """Cut at the first stop string, drop a trailing heading or bold block, and trim spaces."""
    # Cut the raw text at the first stop string, so the next made-up prompt is not kept.
    text = raw
    for stop in GEN_STOPS:
        text = text.split(stop)[0]
    kept = []
    for line in text.splitlines():
        # A heading or a bold block starts a section the model added. Drop it and everything after.
        if line.lstrip().startswith(("#", "**")):
            break
        kept.append(line)
    return "\n".join(kept).strip()


def generate_answer(question, chunk_texts):
    """Greedy answer in the Assignment 1 format, from the 3 numbered chunks."""
    # Fill the template with the question and the 3 chunks, numbered in the order given.
    content = GEN_PROMPT.format(chunk_1=chunk_texts[0], chunk_2=chunk_texts[1], chunk_3=chunk_texts[2], question=question)
    # Wrap the text in the model's chat format, so it is read as a user turn.
    prompt = gen_tok.apply_chat_template([{"role": "user", "content": content}],
                                         tokenize=False, add_generation_prompt=True).rstrip()
    # Turn the prompt into token ids and move them to the GPU.
    ids = gen_tok(prompt, return_tensors="pt").to(gen_model.device)
    with torch.no_grad():
        # Mistral decodes greedily by default (no sampling), so the same answer every run. The length cap ends the answer.
        gen = gen_model.generate(**ids, max_new_tokens=GEN_MAX_NEW_TOKENS,
                                 pad_token_id=gen_tok.eos_token_id)
    # Decode only the new tokens, not the prompt.
    raw = gen_tok.decode(gen[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    # Same handling as Assignment 1 cell 51: keep the text after "Response:" if the model repeats it.
    if "Response:" in raw:
        raw = raw.split("Response:")[1]
    return clean_answer(raw)


def is_not_found(answer):
    """True if the reply is the Not found sentence (a final full stop is allowed)."""
    # Remove spaces and one final full stop, then compare with the exact Not found sentence.
    return answer.strip().rstrip(".").strip() == NOT_FOUND


def has_valid_citation(answer):
    """True if the answer cites at least one of [1], [2], [3] and no other number."""
    # Every number the answer cites. All of them must be 1, 2 or 3 for the citation to count.
    nums = [int(n) for n in CITE.findall(answer)]
    return bool(nums) and all(1 <= n <= 3 for n in nums)
print("generate_answer() defined: greedy answer from the 3 numbered chunks, with the citation and Not found checks.")


#### D1.2 All 10 answers (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: answers query Q1 from its 3 reranked chunks with the Assignment 1 model, then shows the answer.
# Why: D1.2 asks for all 10 answers. One cell per query, so each answer and its citations can be read on its own.
# PDF step: D1.2 (p3). Output: the query, the answer and the chunk numbers it cites.
# ---------------------------------------------------------------------------

answers = {}   # qid -> answer. The 10 query cells below fill it; the judge (D2) uses it.

QID = "Q1"
# Look up this query's text, expected answer and span by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# Generate the answer from this query's 3 reranked chunks. Greedy decoding, so it is repeatable.
answers[QID] = generate_answer(q["query"], [c["text"] for c in final_chunks[QID]])
print(f"{QID}  {q['query']}")
print(f"  Answer: {answers[QID]}")
# Chunk numbers used in the answer, sorted and without repeats. '-' means none.
print(f"  Chunks cited: {','.join(sorted(set(CITE.findall(answers[QID])))) or '-'}")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: answers query Q2 from its 3 reranked chunks with the Assignment 1 model, then shows the answer.
# Why: D1.2 asks for all 10 answers. One cell per query, so each answer and its citations can be read on its own.
# PDF step: D1.2 (p3). Output: the query, the answer and the chunk numbers it cites.
# ---------------------------------------------------------------------------


QID = "Q2"
# Look up this query's text, expected answer and span by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# Generate the answer from this query's 3 reranked chunks. Greedy decoding, so it is repeatable.
answers[QID] = generate_answer(q["query"], [c["text"] for c in final_chunks[QID]])
print(f"{QID}  {q['query']}")
print(f"  Answer: {answers[QID]}")
# Chunk numbers used in the answer, sorted and without repeats. '-' means none.
print(f"  Chunks cited: {','.join(sorted(set(CITE.findall(answers[QID])))) or '-'}")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: answers query Q3 from its 3 reranked chunks with the Assignment 1 model, then shows the answer.
# Why: D1.2 asks for all 10 answers. One cell per query, so each answer and its citations can be read on its own.
# PDF step: D1.2 (p3). Output: the query, the answer and the chunk numbers it cites.
# ---------------------------------------------------------------------------


QID = "Q3"
# Look up this query's text, expected answer and span by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# Generate the answer from this query's 3 reranked chunks. Greedy decoding, so it is repeatable.
answers[QID] = generate_answer(q["query"], [c["text"] for c in final_chunks[QID]])
print(f"{QID}  {q['query']}")
print(f"  Answer: {answers[QID]}")
# Chunk numbers used in the answer, sorted and without repeats. '-' means none.
print(f"  Chunks cited: {','.join(sorted(set(CITE.findall(answers[QID])))) or '-'}")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: answers query Q4 from its 3 reranked chunks with the Assignment 1 model, then shows the answer.
# Why: D1.2 asks for all 10 answers. One cell per query, so each answer and its citations can be read on its own.
# PDF step: D1.2 (p3). Output: the query, the answer and the chunk numbers it cites.
# ---------------------------------------------------------------------------


QID = "Q4"
# Look up this query's text, expected answer and span by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# Generate the answer from this query's 3 reranked chunks. Greedy decoding, so it is repeatable.
answers[QID] = generate_answer(q["query"], [c["text"] for c in final_chunks[QID]])
print(f"{QID}  {q['query']}")
print(f"  Answer: {answers[QID]}")
# Chunk numbers used in the answer, sorted and without repeats. '-' means none.
print(f"  Chunks cited: {','.join(sorted(set(CITE.findall(answers[QID])))) or '-'}")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: answers query Q5 from its 3 reranked chunks with the Assignment 1 model, then shows the answer.
# Why: D1.2 asks for all 10 answers. One cell per query, so each answer and its citations can be read on its own.
# PDF step: D1.2 (p3). Output: the query, the answer and the chunk numbers it cites.
# ---------------------------------------------------------------------------


QID = "Q5"
# Look up this query's text, expected answer and span by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# Generate the answer from this query's 3 reranked chunks. Greedy decoding, so it is repeatable.
answers[QID] = generate_answer(q["query"], [c["text"] for c in final_chunks[QID]])
print(f"{QID}  {q['query']}")
print(f"  Answer: {answers[QID]}")
# Chunk numbers used in the answer, sorted and without repeats. '-' means none.
print(f"  Chunks cited: {','.join(sorted(set(CITE.findall(answers[QID])))) or '-'}")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: answers query Q6 from its 3 reranked chunks with the Assignment 1 model, then shows the answer.
# Why: D1.2 asks for all 10 answers. One cell per query, so each answer and its citations can be read on its own.
# PDF step: D1.2 (p3). Output: the query, the answer and the chunk numbers it cites.
# ---------------------------------------------------------------------------


QID = "Q6"
# Look up this query's text, expected answer and span by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# Generate the answer from this query's 3 reranked chunks. Greedy decoding, so it is repeatable.
answers[QID] = generate_answer(q["query"], [c["text"] for c in final_chunks[QID]])
print(f"{QID}  {q['query']}")
print(f"  Answer: {answers[QID]}")
# Chunk numbers used in the answer, sorted and without repeats. '-' means none.
print(f"  Chunks cited: {','.join(sorted(set(CITE.findall(answers[QID])))) or '-'}")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: answers query Q7 from its 3 reranked chunks with the Assignment 1 model, then shows the answer.
# Why: D1.2 asks for all 10 answers. One cell per query, so each answer and its citations can be read on its own.
# PDF step: D1.2 (p3). Output: the query, the answer and the chunk numbers it cites.
# ---------------------------------------------------------------------------


QID = "Q7"
# Look up this query's text, expected answer and span by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# Generate the answer from this query's 3 reranked chunks. Greedy decoding, so it is repeatable.
answers[QID] = generate_answer(q["query"], [c["text"] for c in final_chunks[QID]])
print(f"{QID}  {q['query']}")
print(f"  Answer: {answers[QID]}")
# Chunk numbers used in the answer, sorted and without repeats. '-' means none.
print(f"  Chunks cited: {','.join(sorted(set(CITE.findall(answers[QID])))) or '-'}")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: answers query Q8 from its 3 reranked chunks with the Assignment 1 model, then shows the answer.
# Why: D1.2 asks for all 10 answers. One cell per query, so each answer and its citations can be read on its own.
# PDF step: D1.2 (p3). Output: the query, the answer and the chunk numbers it cites.
# ---------------------------------------------------------------------------


QID = "Q8"
# Look up this query's text, expected answer and span by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# Generate the answer from this query's 3 reranked chunks. Greedy decoding, so it is repeatable.
answers[QID] = generate_answer(q["query"], [c["text"] for c in final_chunks[QID]])
print(f"{QID}  {q['query']}")
print(f"  Answer: {answers[QID]}")
# Chunk numbers used in the answer, sorted and without repeats. '-' means none.
print(f"  Chunks cited: {','.join(sorted(set(CITE.findall(answers[QID])))) or '-'}")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: answers query Q9 from its 3 reranked chunks with the Assignment 1 model, then shows the answer.
# Why: D1.2 asks for all 10 answers. One cell per query, so each answer and its citations can be read on its own.
# PDF step: D1.2 (p3). Output: the query, the answer and the chunk numbers it cites.
# ---------------------------------------------------------------------------


QID = "Q9"
# Look up this query's text, expected answer and span by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# Generate the answer from this query's 3 reranked chunks. Greedy decoding, so it is repeatable.
answers[QID] = generate_answer(q["query"], [c["text"] for c in final_chunks[QID]])
print(f"{QID}  {q['query']}")
print(f"  Answer: {answers[QID]}")
# Chunk numbers used in the answer, sorted and without repeats. '-' means none.
print(f"  Chunks cited: {','.join(sorted(set(CITE.findall(answers[QID])))) or '-'}")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: answers query Q10 from its 3 reranked chunks with the Assignment 1 model, then shows the answer.
# Why: D1.2 asks for all 10 answers. One cell per query, so each answer and its citations can be read on its own.
# PDF step: D1.2 (p3). Output: the query, the answer and the chunk numbers it cites.
# ---------------------------------------------------------------------------


QID = "Q10"
# Look up this query's text, expected answer and span by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# Generate the answer from this query's 3 reranked chunks. Greedy decoding, so it is repeatable.
answers[QID] = generate_answer(q["query"], [c["text"] for c in final_chunks[QID]])
print(f"{QID}  {q['query']}")
print(f"  Answer: {answers[QID]}")
# Chunk numbers used in the answer, sorted and without repeats. '-' means none.
print(f"  Chunks cited: {','.join(sorted(set(CITE.findall(answers[QID])))) or '-'}")


#### D1.3 A query whose answer is not in the corpus (1 mark)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: runs one out-of-corpus question through the same retrieval, reranking and prompt.
# Why: shows whether the model replies Not found in the documents instead of guessing.
# Output: the question, its 3 chunks, and the reply.
# ---------------------------------------------------------------------------

OOC_QUERY = "Who won the 2018 FIFA World Cup?"   # Our choice: a question the 14 diabetes documents cannot answer
# Same retrieval and reranking as Part C. The middle value is not needed here, so it is _.
_, ooc_top3, _ = rerank_top3(OOC_QUERY)          # same retrieval (best method) and reranking as Part C
for i, c in enumerate(ooc_top3, 1):
    print(f"[{i}] {c['chunk_id']}: {c['text'][:120]}")
# Same generator and prompt as the 10 queries, so the Not found check is fair.
ooc_answer = generate_answer(OOC_QUERY, [c["text"] for c in ooc_top3])
print(f"\nQuestion: {OOC_QUERY}")
print(f"Reply: {ooc_answer}")
# Check the reply with the same exact-match rule used for the 10 answers.
print(f"Replied Not found: {'yes' if is_not_found(ooc_answer) else 'no'}")


#### D1 switch to an instruct model (not used in this notebook)

PDF D1 (p4): "If your Assignment 1 model ignores the chunks or never says 'Not found', you may switch to an instruct model such as Llama 3.1 8B Instruct or Qwen2.5 7B Instruct. Name the model you used and show one example of the problem you saw."

This notebook answers all 10 queries and the out-of-corpus query with the Assignment 1 model only, so no switch is made.


### D2 Check the answers with an LLM judge (4 marks)

**Goal:** grade each answer for correctness against the expected answer and for support by its 3 chunks.

**Judge model:** different from the generator. The judge is `Qwen/Qwen2.5-7B-Instruct`, which differs from the generator (the Assignment 1 Mistral model). Greedy decoding (temperature 0).

**What we do:**
1. D2.1 — run the judge on the 10 answers; per-query table and summary counts.
2. D2.2 — spot-check 3 verdicts by hand.
3. D2.3 — find the cause of one wrong or unsupported answer.
4. D2.4 — suggest one fix.


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: frees the generator and loads the judge model.
# Why: two 7B models do not need to share the GPU at the same time.
# Output: the judge name. The generator is freed before the judge loads.
# ---------------------------------------------------------------------------

import gc

# Free the generator first, so the judge loads into free GPU memory.
del gen_model, gen_base
gc.collect()
torch.cuda.empty_cache()

JUDGE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"   # PDF D2 (p4): "Use a judge model different from your generator, for example Llama 3.1 8B Instruct or Qwen2.5 7B Instruct, and name it." We chose Qwen; it differs from the generator.
# The judge must be a different model from the generator. Checked here, so a mistake stops early.
assert JUDGE_MODEL_ID not in GENERATOR_NAME, "the judge must be a different model from the generator"
with working("Loading the judge"):
    judge_tok = AutoTokenizer.from_pretrained(JUDGE_MODEL_ID)
    judge_model = AutoModelForCausalLM.from_pretrained(JUDGE_MODEL_ID, device_map={"": 0}).eval()
# Qwen's own generation config sets sampling values. Greedy decoding ignores them, so clear them (no warnings).
judge_model.generation_config.temperature = None
judge_model.generation_config.top_p = None
judge_model.generation_config.top_k = None


def judge_generate(prompt_text):
    """Greedy judge reply (temperature 0), using the judge's own chat template."""
    text = judge_tok.apply_chat_template([{"role": "user", "content": prompt_text}],
                                         tokenize=False, add_generation_prompt=True)
    ids = judge_tok(text, return_tensors="pt").to(judge_model.device)
    with torch.no_grad():
        out = judge_model.generate(**ids, max_new_tokens=120, do_sample=False, pad_token_id=judge_tok.eos_token_id)
    return judge_tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)


print(f"judge: {JUDGE_MODEL_ID}")


#### D2.1 Judge prompt and results (1 mark)


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines the judge prompt and prints it.
# Why: the brief asks to show the prompt; it follows every hint in the brief (D2.1, p4).
# Output: the judge prompt.
# ---------------------------------------------------------------------------

JUDGE_PROMPT = """You are grading one answer about Type 2 Diabetes.

Question: {question}
Expected answer (reference): {expected}
Retrieved chunks:
[1] {chunk_1}
[2] {chunk_2}
[3] {chunk_3}
Model answer: {answer}

Check 1, CORRECT: compare the model answer with the expected answer.
  CORRECT = it states the same key fact as the expected answer.
  PARTLY CORRECT = it states only part of the key fact, or the right fact with a detail missing or wrong.
  WRONG = it states a different fact, contradicts the expected answer, or says "Not found in the documents".
Check 2, SUPPORTED: can every statement in the model answer be found in the 3 chunks above?
  YES = every statement is in the chunks. NO = at least one statement is not.
  A "Not found in the documents" reply makes no claims, so it is SUPPORTED: YES.
  A reply of exactly "Not found in the documents" always gets CORRECT: WRONG and SUPPORTED: YES.
REASON: one sentence. If SUPPORTED is NO, the reason must quote the unsupported statement in double quotes.

Reply in exactly this format, with nothing else:
CORRECT: <CORRECT | PARTLY CORRECT | WRONG>
SUPPORTED: <YES | NO>
REASON: <one sentence>"""
print(JUDGE_PROMPT)

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: defines parse_verdict(), which reads the judge's three labelled lines, and starts the verdicts.
# Why: the judge replies in the fixed format from D2.1, so code can count the verdicts. Each query is judged in its own cell.
# PDF step: D2.1 (p4). Output: nothing (definitions only).
# ---------------------------------------------------------------------------


def parse_verdict(text):
    """Read the CORRECT, SUPPORTED and REASON labels. The <> brackets and ** marks copied from the template are removed first."""
    # Remove the <> and ** marks the judge copies from the template, so '<CORRECT>' reads as CORRECT.
    flat = re.sub(r"[<>*]", "", text)

    # Read the value after 'label:' up to the next label or the end of the line.
    def pick(label):
        m = re.search(rf"{label}\s*:\s*(.*?)(?=\s+(?:CORRECT|SUPPORTED|REASON)\s*:|\n|$)", flat, re.S | re.I)
        return " ".join(m.group(1).split()).rstrip(".") if m else ""
    # Only values in the allowed lists count. Anything else is marked UNPARSED.
    correct = pick("CORRECT").upper()
    supported = pick("SUPPORTED").upper()
    # A value that is not in the required format is marked UNPARSED and is not counted in the summary.
    return {"correct": correct if correct in ("CORRECT", "PARTLY CORRECT", "WRONG") else "UNPARSED",
            "supported": supported if supported in ("YES", "NO") else "UNPARSED",
            "reason": pick("REASON")}


verdicts = {}   # qid -> verdict, filled by the 10 query cells below
print("parse_verdict() defined. The judge cells below fill verdicts one query at a time.")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: judges query Q1 with the judge prompt and reads its verdict.
# Why: D2.1 asks for the judge on all 10 answers. One cell per query, so each verdict is shown on its own.
# PDF step: D2.1 (p4). Output: the query, CORRECT, SUPPORTED and REASON.
# ---------------------------------------------------------------------------

QID = "Q1"
# Look up this query's text, expected answer and span by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# The 3 chunks the generator saw, in order, for the support check.
texts = [c["text"] for c in final_chunks[QID]]
prompt = JUDGE_PROMPT.format(question=q["query"], expected=q["expected_answer"], chunk_1=texts[0],
                             chunk_2=texts[1], chunk_3=texts[2], answer=answers[QID])
# Greedy judge reply (temperature 0), so the same answer gets the same verdict.
raw = judge_generate(prompt)
# Read the judge's three labelled lines. Anything outside the format is marked UNPARSED.
verdicts[QID] = dict(parse_verdict(raw), raw=raw)
v = verdicts[QID]
print(f"{QID}  {q['query']}")
print(f"  CORRECT={v['correct']}  SUPPORTED={v['supported']}  REASON={v['reason']}")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: judges query Q2 with the judge prompt and reads its verdict.
# Why: D2.1 asks for the judge on all 10 answers. One cell per query, so each verdict is shown on its own.
# PDF step: D2.1 (p4). Output: the query, CORRECT, SUPPORTED and REASON.
# ---------------------------------------------------------------------------

QID = "Q2"
# Look up this query's text, expected answer and span by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# The 3 chunks the generator saw, in order, for the support check.
texts = [c["text"] for c in final_chunks[QID]]
prompt = JUDGE_PROMPT.format(question=q["query"], expected=q["expected_answer"], chunk_1=texts[0],
                             chunk_2=texts[1], chunk_3=texts[2], answer=answers[QID])
# Greedy judge reply (temperature 0), so the same answer gets the same verdict.
raw = judge_generate(prompt)
# Read the judge's three labelled lines. Anything outside the format is marked UNPARSED.
verdicts[QID] = dict(parse_verdict(raw), raw=raw)
v = verdicts[QID]
print(f"{QID}  {q['query']}")
print(f"  CORRECT={v['correct']}  SUPPORTED={v['supported']}  REASON={v['reason']}")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: judges query Q3 with the judge prompt and reads its verdict.
# Why: D2.1 asks for the judge on all 10 answers. One cell per query, so each verdict is shown on its own.
# PDF step: D2.1 (p4). Output: the query, CORRECT, SUPPORTED and REASON.
# ---------------------------------------------------------------------------

QID = "Q3"
# Look up this query's text, expected answer and span by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# The 3 chunks the generator saw, in order, for the support check.
texts = [c["text"] for c in final_chunks[QID]]
prompt = JUDGE_PROMPT.format(question=q["query"], expected=q["expected_answer"], chunk_1=texts[0],
                             chunk_2=texts[1], chunk_3=texts[2], answer=answers[QID])
# Greedy judge reply (temperature 0), so the same answer gets the same verdict.
raw = judge_generate(prompt)
# Read the judge's three labelled lines. Anything outside the format is marked UNPARSED.
verdicts[QID] = dict(parse_verdict(raw), raw=raw)
v = verdicts[QID]
print(f"{QID}  {q['query']}")
print(f"  CORRECT={v['correct']}  SUPPORTED={v['supported']}  REASON={v['reason']}")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: judges query Q4 with the judge prompt and reads its verdict.
# Why: D2.1 asks for the judge on all 10 answers. One cell per query, so each verdict is shown on its own.
# PDF step: D2.1 (p4). Output: the query, CORRECT, SUPPORTED and REASON.
# ---------------------------------------------------------------------------

QID = "Q4"
# Look up this query's text, expected answer and span by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# The 3 chunks the generator saw, in order, for the support check.
texts = [c["text"] for c in final_chunks[QID]]
prompt = JUDGE_PROMPT.format(question=q["query"], expected=q["expected_answer"], chunk_1=texts[0],
                             chunk_2=texts[1], chunk_3=texts[2], answer=answers[QID])
# Greedy judge reply (temperature 0), so the same answer gets the same verdict.
raw = judge_generate(prompt)
# Read the judge's three labelled lines. Anything outside the format is marked UNPARSED.
verdicts[QID] = dict(parse_verdict(raw), raw=raw)
v = verdicts[QID]
print(f"{QID}  {q['query']}")
print(f"  CORRECT={v['correct']}  SUPPORTED={v['supported']}  REASON={v['reason']}")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: judges query Q5 with the judge prompt and reads its verdict.
# Why: D2.1 asks for the judge on all 10 answers. One cell per query, so each verdict is shown on its own.
# PDF step: D2.1 (p4). Output: the query, CORRECT, SUPPORTED and REASON.
# ---------------------------------------------------------------------------

QID = "Q5"
# Look up this query's text, expected answer and span by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# The 3 chunks the generator saw, in order, for the support check.
texts = [c["text"] for c in final_chunks[QID]]
prompt = JUDGE_PROMPT.format(question=q["query"], expected=q["expected_answer"], chunk_1=texts[0],
                             chunk_2=texts[1], chunk_3=texts[2], answer=answers[QID])
# Greedy judge reply (temperature 0), so the same answer gets the same verdict.
raw = judge_generate(prompt)
# Read the judge's three labelled lines. Anything outside the format is marked UNPARSED.
verdicts[QID] = dict(parse_verdict(raw), raw=raw)
v = verdicts[QID]
print(f"{QID}  {q['query']}")
print(f"  CORRECT={v['correct']}  SUPPORTED={v['supported']}  REASON={v['reason']}")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: judges query Q6 with the judge prompt and reads its verdict.
# Why: D2.1 asks for the judge on all 10 answers. One cell per query, so each verdict is shown on its own.
# PDF step: D2.1 (p4). Output: the query, CORRECT, SUPPORTED and REASON.
# ---------------------------------------------------------------------------

QID = "Q6"
# Look up this query's text, expected answer and span by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# The 3 chunks the generator saw, in order, for the support check.
texts = [c["text"] for c in final_chunks[QID]]
prompt = JUDGE_PROMPT.format(question=q["query"], expected=q["expected_answer"], chunk_1=texts[0],
                             chunk_2=texts[1], chunk_3=texts[2], answer=answers[QID])
# Greedy judge reply (temperature 0), so the same answer gets the same verdict.
raw = judge_generate(prompt)
# Read the judge's three labelled lines. Anything outside the format is marked UNPARSED.
verdicts[QID] = dict(parse_verdict(raw), raw=raw)
v = verdicts[QID]
print(f"{QID}  {q['query']}")
print(f"  CORRECT={v['correct']}  SUPPORTED={v['supported']}  REASON={v['reason']}")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: judges query Q7 with the judge prompt and reads its verdict.
# Why: D2.1 asks for the judge on all 10 answers. One cell per query, so each verdict is shown on its own.
# PDF step: D2.1 (p4). Output: the query, CORRECT, SUPPORTED and REASON.
# ---------------------------------------------------------------------------

QID = "Q7"
# Look up this query's text, expected answer and span by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# The 3 chunks the generator saw, in order, for the support check.
texts = [c["text"] for c in final_chunks[QID]]
prompt = JUDGE_PROMPT.format(question=q["query"], expected=q["expected_answer"], chunk_1=texts[0],
                             chunk_2=texts[1], chunk_3=texts[2], answer=answers[QID])
# Greedy judge reply (temperature 0), so the same answer gets the same verdict.
raw = judge_generate(prompt)
# Read the judge's three labelled lines. Anything outside the format is marked UNPARSED.
verdicts[QID] = dict(parse_verdict(raw), raw=raw)
v = verdicts[QID]
print(f"{QID}  {q['query']}")
print(f"  CORRECT={v['correct']}  SUPPORTED={v['supported']}  REASON={v['reason']}")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: judges query Q8 with the judge prompt and reads its verdict.
# Why: D2.1 asks for the judge on all 10 answers. One cell per query, so each verdict is shown on its own.
# PDF step: D2.1 (p4). Output: the query, CORRECT, SUPPORTED and REASON.
# ---------------------------------------------------------------------------

QID = "Q8"
# Look up this query's text, expected answer and span by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# The 3 chunks the generator saw, in order, for the support check.
texts = [c["text"] for c in final_chunks[QID]]
prompt = JUDGE_PROMPT.format(question=q["query"], expected=q["expected_answer"], chunk_1=texts[0],
                             chunk_2=texts[1], chunk_3=texts[2], answer=answers[QID])
# Greedy judge reply (temperature 0), so the same answer gets the same verdict.
raw = judge_generate(prompt)
# Read the judge's three labelled lines. Anything outside the format is marked UNPARSED.
verdicts[QID] = dict(parse_verdict(raw), raw=raw)
v = verdicts[QID]
print(f"{QID}  {q['query']}")
print(f"  CORRECT={v['correct']}  SUPPORTED={v['supported']}  REASON={v['reason']}")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: judges query Q9 with the judge prompt and reads its verdict.
# Why: D2.1 asks for the judge on all 10 answers. One cell per query, so each verdict is shown on its own.
# PDF step: D2.1 (p4). Output: the query, CORRECT, SUPPORTED and REASON.
# ---------------------------------------------------------------------------

QID = "Q9"
# Look up this query's text, expected answer and span by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# The 3 chunks the generator saw, in order, for the support check.
texts = [c["text"] for c in final_chunks[QID]]
prompt = JUDGE_PROMPT.format(question=q["query"], expected=q["expected_answer"], chunk_1=texts[0],
                             chunk_2=texts[1], chunk_3=texts[2], answer=answers[QID])
# Greedy judge reply (temperature 0), so the same answer gets the same verdict.
raw = judge_generate(prompt)
# Read the judge's three labelled lines. Anything outside the format is marked UNPARSED.
verdicts[QID] = dict(parse_verdict(raw), raw=raw)
v = verdicts[QID]
print(f"{QID}  {q['query']}")
print(f"  CORRECT={v['correct']}  SUPPORTED={v['supported']}  REASON={v['reason']}")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: judges query Q10 with the judge prompt and reads its verdict.
# Why: D2.1 asks for the judge on all 10 answers. One cell per query, so each verdict is shown on its own.
# PDF step: D2.1 (p4). Output: the query, CORRECT, SUPPORTED and REASON.
# ---------------------------------------------------------------------------

QID = "Q10"
# Look up this query's text, expected answer and span by its id.
q = next(x for x in QUERIES if x["qid"] == QID)
# The 3 chunks the generator saw, in order, for the support check.
texts = [c["text"] for c in final_chunks[QID]]
prompt = JUDGE_PROMPT.format(question=q["query"], expected=q["expected_answer"], chunk_1=texts[0],
                             chunk_2=texts[1], chunk_3=texts[2], answer=answers[QID])
# Greedy judge reply (temperature 0), so the same answer gets the same verdict.
raw = judge_generate(prompt)
# Read the judge's three labelled lines. Anything outside the format is marked UNPARSED.
verdicts[QID] = dict(parse_verdict(raw), raw=raw)
v = verdicts[QID]
print(f"{QID}  {q['query']}")
print(f"  CORRECT={v['correct']}  SUPPORTED={v['supported']}  REASON={v['reason']}")


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: shows the D2.1 results table for all 10 queries and counts the verdicts.
# Why: the brief asks for one table with a row per query, and a summary out of 10.
# PDF step: D2.1 (p4). Output: the table, the summary counts and any reply the judge did not follow.
# ---------------------------------------------------------------------------

query_text = {q["qid"]: q["query"] for q in QUERIES}
# Verdicts marked UNPARSED in either check are left out of the counts.
unparsed = [qid for qid, v in verdicts.items() if "UNPARSED" in (v["correct"], v["supported"])]
show_md_table(["Query", "Correct", "Supported", "Reason"],
              [(f"{qid}: {query_text[qid]}", v["correct"], v["supported"], v["reason"]) for qid, v in verdicts.items()],
              title="D2.1 judge results", save_as="d2_verdicts")
# Count how many queries got each correctness label.
counts = {k: sum(v["correct"] == k for v in verdicts.values()) for k in ("CORRECT", "PARTLY CORRECT", "WRONG")}
print_table(["Check", "Count out of 10"], [
    ("Correct", str(counts["CORRECT"])),
    ("Partly correct", str(counts["PARTLY CORRECT"])),
    ("Wrong", str(counts["WRONG"])),
    ("Supported by the chunks", str(sum(v["supported"] == "YES" for v in verdicts.values()))),
], title="D2.1 summary", save_as="d2_summary")
# List the replies that did not follow the format, so the reader can see them.
print(f"lines the judge did not follow the format: {len(unparsed)} {unparsed}")
for qid in unparsed:
    print(f"  {qid} judge reply: {' '.join(verdicts[qid]['raw'].split())[:300]}")


#### D2.2 Spot-check the judge (1 mark)

*The cell below shows 3 answers (with different verdicts where possible), their 3 chunks and the judge's verdicts. Read them, then fill in the notes cell after it.*

In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: shows the first 3 answers with their chunks and the judge's verdicts, for your own reading.
# Why: PDF D2.2 asks you to read 3 answers and their chunks yourself, then say how many verdicts you agree with.
# PDF step: D2.2 (p4). Output: the 3 answers, their chunks and the judge's verdicts.
# ---------------------------------------------------------------------------

for q in QUERIES[:3]:
    qid = q["qid"]
    print(f"\n{qid}  {q['query']}")
    print(f"  Expected answer: {q['expected_answer']}")
    print(f"  Model answer: {answers[qid]}")
    for i, c in enumerate(final_chunks[qid], 1):
        print(f"  [{i}] {c['text']}")
    print(f"  Judge: CORRECT={verdicts[qid]['correct']}  SUPPORTED={verdicts[qid]['supported']}  REASON={verdicts[qid]['reason']}")


**D2.2 my spot-check** *(fill in after reading the 3 answers above)*

| Query | Judge (correct / supported) | Mine (correct / supported) | Agree? |
| --- | --- | --- | --- |
| Q_ | | | |
| Q_ | | | |
| Q_ | | | |

I agree with _ of the 3 verdicts. Disagreement (if any): _one or two sentences on which verdict and why._

#### D2.3 Find the cause (1 mark)

*The cell below picks one answer marked WRONG or SUPPORTED: NO, shows its 3 chunks, and checks whether the answer span is in them: was the needed information in the chunks or not?*


In [ ]:
# ---------------------------------------------------------------------------
# What this cell does: picks the first answer marked WRONG or SUPPORTED: NO, shows its 3 chunks and checks the span.
# Why: PDF D2.3 asks for one such answer, its 3 chunks, and whether the needed information was in the chunks.
# PDF step: D2.3 (p5). Output: the answer, its chunks, and whether the answer span is in them.
# ---------------------------------------------------------------------------


def failed(qid):
    """True if the judge marked the answer WRONG on correctness or NO on support."""
    return verdicts[qid]["correct"] == "WRONG" or verdicts[qid]["supported"] == "NO"


FAILED_QID = next((q["qid"] for q in QUERIES if failed(q["qid"])), None)
if FAILED_QID is None:
    print("No answer is WRONG or unsupported, so there is nothing to analyse.")
else:
    q = next(x for x in QUERIES if x["qid"] == FAILED_QID)
    print(f"{FAILED_QID}  {q['query']}")
    print(f"  Expected answer: {q['expected_answer']}")
    print(f"  Model answer: {answers[FAILED_QID]}")
    print(f"  Judge: CORRECT={verdicts[FAILED_QID]['correct']}  SUPPORTED={verdicts[FAILED_QID]['supported']}  "
          f"REASON={verdicts[FAILED_QID]['reason']}")
    for i, c in enumerate(final_chunks[FAILED_QID], 1):
        print(f"\n  [{i}] {c['chunk_id']}:\n{c['text']}")
    # Was the needed information in the 3 chunks? The answer span is the needed information.
    in_chunks = any(is_hit(c["text"], q["span"]) for c in final_chunks[FAILED_QID])
    print(f"\n  Answer span in the 3 chunks: {'yes' if in_chunks else 'no'}")


**D2.3 cause in my words** *(fill in after reading the 3 chunks above)*: _was the needed information in the chunks or not, and what did the model do with it? One or two sentences._

#### D2.4 Suggest a fix (1 mark)

*One change that matches the cause found in D2.3, and why it would help.*